# Experiment: SpectralBio Final Accept Hardening Part 3 (A100)

This notebook is the killer-evidence notebook for the revision.

## Scope
- Exact-split ESM-1v extraction on TP53, BRCA1, BRCA2, and NSD1
- Test whether covariance adds signal on top of ESM-1v
- Run matched permutation/null audits to kill the rescaling-artifact hypothesis
- Optionally run a 3B shadow augmentation on TP53 and BRCA2

## Intended runtime
- Target hardware: A100
- This is the highest-ROI notebook in the whole campaign


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_OUTPUT_SUBDIR = Path('MyDrive/SpectralBioFinalAcceptA100')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
    print('Drive mounted at', DRIVE_MOUNT_POINT)
else:
    print('Drive mount skipped; outputs stay in the VM unless OUTPUT_ROOT is changed later.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/DaviBonetto/SpectralBio.git'
REPO_BRANCH = 'codex/claw4s-rebuild'
DEFAULT_REPO_ROOT = Path('/content/Stanford-Claw4s')
ENV_REPO_ROOT = os.environ.get('SPECTRALBIO_REPO_ROOT', '').strip()


def _looks_like_repo(path: Path) -> bool:
    return (path / 'src' / 'spectralbio').exists() and (path / 'notebooks').exists()


candidate_roots = []
if ENV_REPO_ROOT:
    candidate_roots.append(Path(ENV_REPO_ROOT).expanduser())
candidate_roots.extend([Path.cwd(), Path.cwd().parent, DEFAULT_REPO_ROOT])

REPO_ROOT = next((path.resolve() for path in candidate_roots if _looks_like_repo(path)), DEFAULT_REPO_ROOT)
BOOTSTRAP_MARKER = REPO_ROOT / '.colab_bootstrap_v4_complete'

if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.check_call(['git', 'fetch', 'origin', REPO_BRANCH])
subprocess.check_call(['git', 'checkout', REPO_BRANCH])
src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if not BOOTSTRAP_MARKER.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'numpy==2.1.3', 'plotly==5.24.1', 'pyyaml==6.0.2', 'scikit-learn==1.5.2',
        'scipy==1.14.1', 'transformers==4.49.0', 'accelerate>=1.0.0', 'pandas', 'matplotlib'
    ])
    BOOTSTRAP_MARKER.write_text('ok\n', encoding='utf-8')
    print('Dependencies installed. Continuing in the same runtime.')
else:
    print('Bootstrap marker found; skipping reinstall.')


## Plan

- Rebuild the exact external baseline surface instead of relying only on quoted numbers.
- Test additive value, not headline replacement.
- Use permutation controls so a reviewer cannot dismiss the covariance term as monotonic noise.
- Treat TP53 as the primary target and BRCA2 as the external extension target.


In [ ]:
import subprocess
import sys
import pandas as pd

try:
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        run_esm1v_augmentation_suite,
        run_esm1v_permutation_audit,
    )
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        run_esm1v_augmentation_suite,
        run_esm1v_permutation_audit,
    )

RUN_GENES = ('TP53', 'BRCA1', 'BRCA2', 'NSD1')
PRIMARY_GENES = ('TP53', 'BRCA2')
ESM1V_MODEL_IDS = tuple(f'facebook/esm1v_t33_650M_UR90S_{i}' for i in range(1, 6))
PERMUTATION_REPLICATES = 1000
RUN_3B_SHADOW = True
OVERWRITE = False

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / 'supplementary' / 'final_accept_a100'
)

config = AcceptHardeningConfig(
    stronger_model_names=('facebook/esm2_t33_650M_UR50D', 'facebook/esm2_t36_3B_UR50D'),
    large_model_gene_limit=4,
    overwrite=OVERWRITE,
)
paths = create_accept_hardening_paths(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)
print(paths)
print(config)
print('ESM-1v models:', ESM1V_MODEL_IDS)



In [ ]:
augmentation_summary = run_esm1v_augmentation_suite(
    paths=paths,
    config=config,
    genes=RUN_GENES,
    esm1v_model_ids=ESM1V_MODEL_IDS,
    primary_genes=PRIMARY_GENES,
    run_3b_shadow=RUN_3B_SHADOW,
)
permutation_summary = run_esm1v_permutation_audit(
    paths=paths,
    config=config,
    genes=PRIMARY_GENES,
    permutation_replicates=PERMUTATION_REPLICATES,
)
display(pd.DataFrame(augmentation_summary['gene_rows']))
display(pd.DataFrame(permutation_summary['rows']))



In [ ]:
import shutil
from IPython.display import FileLink, HTML, Javascript, display

expected_paths = [
    paths.root / 'esm1v_augmentation' / 'esm1v_augmentation_summary.json',
    paths.root / 'esm1v_augmentation' / 'esm1v_augmentation_long.csv',
    paths.root / 'esm1v_augmentation' / 'esm1v_permutation_audit.csv',
]
for path in expected_paths:
    print(path)

source_dir = paths.root / 'esm1v_augmentation'
if not source_dir.exists():
    raise FileNotFoundError(f'Missing expected output directory: {source_dir}')

part3_bundle_root = paths.root / '_downloads' / 'part3_esm1v_augmentation'
if part3_bundle_root.exists():
    shutil.rmtree(part3_bundle_root)
part3_bundle_root.mkdir(parents=True, exist_ok=True)
shutil.copytree(source_dir, part3_bundle_root / 'esm1v_augmentation', dirs_exist_ok=True)

manifest_lines = [
    'Part 3 bundled outputs',
    f'Results root: {paths.root}',
    f'Generated at: {pd.Timestamp.utcnow()}',
    '',
    'Included folders:',
    '- esm1v_augmentation',
]
(part3_bundle_root / 'bundle_manifest.txt').write_text('\n'.join(manifest_lines) + '\n', encoding='utf-8')

archive_path = REPO_ROOT / 'notebooks' / 'final_accept_part3_esm1v_augmentation_results.zip'
if archive_path.exists():
    archive_path.unlink()
archive_base = archive_path.with_suffix('')
shutil.make_archive(str(archive_base), 'zip', root_dir=part3_bundle_root.parent, base_dir=part3_bundle_root.name)
print('ZIP ready:', archive_path)
display(FileLink(str(archive_path), result_html_prefix='Download ZIP: '))

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    download_href_candidates = [
        f'/files/notebooks/{archive_path.name}',
        f'files/notebooks/{archive_path.name}',
        archive_path.name,
    ]
    links_html = ''.join(
        f'<li><a class="part3-download-link" href="{href}" download>{href}</a></li>'
        for href in download_href_candidates
    )
    display(HTML(
        '<div>'
        '<p>If automatic download is blocked, click one of the links below.</p>'
        f'<ul>{links_html}</ul>'
        '</div>'
    ))
    js = """
(() => {
  const links = Array.from(document.querySelectorAll('.part3-download-link'));
  const preferred = links.find((link) => link.getAttribute('href').startsWith('/files/')) || links[0];
  if (preferred) {
    preferred.click();
  }
})();
"""
    display(Javascript(js))


In [ ]:
print('Part 3 notebook complete.')
print('Output root:', paths.root)
if 'archive_path' in globals():
    print('ZIP archive:', archive_path)
else:
    print('Run the previous cell to build the zip archive.')
